In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!pip install pandas
!pip install numpy
!pip install opencv-python
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn
!pip install tensorflow==2.9.2
!pip install keras
!pip install albumentations
!pip install tqdm

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import time
import shutil
import random
from collections import Counter
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras import regularizers
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras.applications import EfficientNetV2B0, EfficientNetV2B1, EfficientNetV2B2, EfficientNetV2B3, EfficientNetV2S, EfficientNetV2M, EfficientNetV2L
import albumentations as A
random.seed(123)
np.random.seed(123)
tf.random.set_seed(123)
print('tensorflow version is ', tf.__version__)

In [ ]:
def print_in_color(txt_msg, fore_tupple=(0, 255, 255), back_tupple=(0, 0, 0)):
    rf, gf, bf = fore_tupple
    rb, gb, bb = back_tupple
    mat = '\33[38;2;' + str(rf) + ';' + str(gf) + ';' + str(bf) + ';48;2;' + str(rb) + ';' + str(gb) + ';' + str(bb) + 'm'
    print(mat + txt_msg)
    print('\33[0m', end='')


def ask_int(default=None, allow_empty=False, minimum=None, maximum=None):
    while True:
        ans = input(' ').strip()
        if ans == '' and allow_empty:
            return default
        try:
            value = int(ans)
        except ValueError:
            print_in_color('please enter an integer')
            continue
        if minimum is not None and value < minimum:
            print_in_color(f'the value must be at least {minimum}')
            continue
        if maximum is not None and value > maximum:
            print_in_color(f'the value must be at most {maximum}')
            continue
        return value

In [ ]:
def plot_label_count(df, plot_title):
    column = 'labels'
    xaxis_label = 'CLASS'
    yaxis_label = 'IMAGE COUNT'
    vcounts = df[column].value_counts()
    labels = vcounts.keys().tolist()
    values = vcounts.tolist()
    lcount = len(labels)
    if lcount > 55:
        print_in_color('The number of labels is >55, no plot will be produced')
    else:
        width = np.min([lcount * 4, 20])
        plt.figure(figsize=(width, 5))
        form = {'family': 'serif', 'color': 'blue', 'size': 25}
        plt.bar(labels, values)
        plt.title(plot_title, fontsize=24, color='blue')
        plt.xticks(rotation=90, fontsize=18)
        plt.yticks(fontsize=18)
        plt.xlabel(xaxis_label, fontdict=form)
        plt.ylabel(yaxis_label, fontdict=form)
        if lcount >= 8:
            rotation = 'vertical'
        else:
            rotation = 'horizontal'
        for i in range(lcount):
            plt.text(i, values[i] / 2, str(values[i]), fontsize=12, rotation=rotation, color='yellow', ha='center')
        plt.show()

In [ ]:
def check_dataset_size(train_dir):
    classes = sorted(os.listdir(train_dir))
    ftotal = 0
    flargest = 0
    fsmallest = float('inf')
    maxclass = None
    minclass = None
    for klass in classes:
        classpath = os.path.join(train_dir, klass)
        if os.path.isdir(classpath):
            fcount = len(os.listdir(classpath))
            if fcount > flargest:
                flargest = fcount
                maxclass = klass
            if fcount < fsmallest:
                fsmallest = fcount
                minclass = klass
            ftotal += fcount
    if maxclass is None:
        raise ValueError('no class directories were found in the train directory')
    return ftotal, flargest, maxclass, fsmallest, minclass

In [ ]:
def make_dataframes(train_dir, test_dir, val_dir, limiter, min_images):
    bad_images = []
    skip_classes = []
    if test_dir is None and val_dir is None:
        dirlist = [train_dir]
        names = ['train']
    elif test_dir is None:
        dirlist = [train_dir, val_dir]
        names = ['train', 'valid']
    elif val_dir is None:
        dirlist = [train_dir, test_dir]
        names = ['train', 'test']
    else:
        dirlist = [train_dir, test_dir, val_dir]
        names = ['train', 'test', 'valid']
    ht = 0
    wt = 0
    total_good_files = 0
    train_df = None
    test_df = None
    valid_df = None
    for name, d in zip(names, dirlist):
        filepaths = []
        labels = []
        for klass in sorted(os.listdir(d)):
            print(f'processing images in {name} directory for class {klass}' + ' ' * 50, '\r', end='')
            classpath = os.path.join(d, klass)
            if not os.path.isdir(classpath):
                continue
            flist = sorted(os.listdir(classpath))
            if name == 'train' and len(flist) < min_images:
                skip_classes.append(klass)
                continue
            if klass in skip_classes:
                continue
            if limiter is not None and limiter < len(flist):
                flist = list(np.random.choice(flist, limiter, replace=False))
            for f in flist:
                fpath = os.path.join(classpath, f)
                ext = os.path.splitext(f)[1][1:].lower()
                if ext not in ['jpg', 'jpeg', 'tiff', 'png', 'bmp']:
                    bad_images.append(fpath)
                    continue
                img = cv2.imread(fpath)
                if img is None:
                    bad_images.append(fpath)
                    continue
                ht += img.shape[0]
                wt += img.shape[1]
                total_good_files += 1
                filepaths.append(fpath)
                labels.append(klass)
        print('')
        df = pd.DataFrame({'filepaths': filepaths, 'labels': labels})
        if name == 'valid':
            valid_df = df
        elif name == 'test':
            test_df = df
        else:
            if test_dir is None and val_dir is None:
                train_df, dummy_df = train_test_split(df, train_size=.8, shuffle=True, random_state=123, stratify=df['labels'])
                valid_df, test_df = train_test_split(dummy_df, train_size=.5, shuffle=True, random_state=123, stratify=dummy_df['labels'])
            elif test_dir is None:
                train_df, test_df = train_test_split(df, train_size=.8, shuffle=True, random_state=123, stratify=df['labels'])
            elif val_dir is None:
                train_df, valid_df = train_test_split(df, train_size=.8, shuffle=True, random_state=123, stratify=df['labels'])
            else:
                train_df = df
    if total_good_files == 0:
        raise ValueError('no valid image files were found')
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    classes = sorted(train_df['labels'].unique())
    class_count = len(classes)
    have = int(ht / total_good_files)
    wave = int(wt / total_good_files)
    print('number of classes in processed dataset= ', class_count)
    if len(skip_classes) > 0:
        print_in_color(f'listed below are classes not included in the dataset because it had less than {min_images} training image files')
        for klass in skip_classes:
            print(klass)
    counts = list(train_df['labels'].value_counts())
    print('the maximum files in any class in train_df is ', max(counts), '  the minimum files in any class in train_df is ', min(counts))
    print('train_df length: ', len(train_df), '  test_df length: ', len(test_df), '  valid_df length: ', len(valid_df))
    if len(bad_images) == 0:
        print_in_color('All image files were properly processed and used in the dataframes')
    else:
        print_in_color(f'there are {len(bad_images)} bad image files and {total_good_files} proper image files in the dataset')
        for f in bad_images:
            print(f)
    plot_label_count(train_df, 'Images per Label in train set')
    return train_df, test_df, valid_df, classes, class_count, max(counts), min(counts), have, wave

In [ ]:
def trim(df, max_samples, min_samples, column):
    df = df.copy()
    class_count = df[column].nunique()
    print('dataframe initially is of length ', len(df), ' with ', class_count, ' classes')
    parts = []
    for label, group in df.groupby(column):
        count = len(group)
        if count > max_samples:
            parts.append(group.sample(n=max_samples, random_state=123))
        elif count >= min_samples:
            parts.append(group)
    if len(parts) == 0:
        raise ValueError('no class has enough images after trimming')
    trimmed_df = pd.concat(parts, axis=0).reset_index(drop=True)
    counts = trimmed_df[column].value_counts()
    print('after trimming, the maximum samples in any class is now ', counts.max(), ' and the minimum samples in any class is ', counts.min())
    classes = trimmed_df[column].unique()
    class_count = len(classes)
    print('the trimmed dataframe now is of length ', len(trimmed_df), ' with ', class_count, ' classes')
    return trimmed_df, classes, class_count

In [ ]:
def balance(df, n, column, working_dir):
    def get_augmented_image(image):
        width = int(image.shape[1] * .8)
        height = int(image.shape[0] * .8)
        transform = A.Compose([
            A.HorizontalFlip(p=.5),
            A.Rotate(limit=30, p=.25),
            A.RandomBrightnessContrast(p=.5),
            A.RandomGamma(p=.5),
            A.RandomCrop(width=width, height=height, p=.25)])
        return transform(image=image)['image']

    df = df.copy()
    print('Initial length of dataframe is ', len(df))
    aug_dir = os.path.join(working_dir, 'aug')
    if os.path.isdir(aug_dir):
        shutil.rmtree(aug_dir)
    os.makedirs(aug_dir)
    for label in df[column].unique():
        os.makedirs(os.path.join(aug_dir, label), exist_ok=True)
    total = 0
    groups = df.groupby(column)
    for label in df[column].unique():
        print(f'augmenting images in train set for class {label}' + ' ' * 50, '\r', end='')
        group = groups.get_group(label)
        sample_count = len(group)
        if sample_count < n:
            delta = n - sample_count
            target_dir = os.path.join(aug_dir, label)
            for i in range(delta):
                j = i % sample_count
                img_path = group['filepaths'].iloc[j]
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = get_augmented_image(img)
                fname = 'aug' + str(i) + '-' + os.path.basename(img_path)
                cv2.imwrite(os.path.join(target_dir, fname), img)
                total += 1
    print('')
    print('Total Augmented images created= ', total)
    aug_fpaths = []
    aug_labels = []
    for klass in sorted(os.listdir(aug_dir)):
        classpath = os.path.join(aug_dir, klass)
        for f in sorted(os.listdir(classpath)):
            aug_fpaths.append(os.path.join(classpath, f))
            aug_labels.append(klass)
    aug_df = pd.DataFrame({'filepaths': aug_fpaths, 'labels': aug_labels})
    df = pd.concat([df, aug_df], axis=0).reset_index(drop=True)
    print('Length of augmented dataframe is now ', len(df))
    return df

In [ ]:
def make_gens(batch_size, ycol, train_df, test_df, valid_df, img_size, classes):
    classes = list(classes)
    valid_df = valid_df[valid_df[ycol].isin(classes)].reset_index(drop=True)
    test_df = test_df[test_df[ycol].isin(classes)].reset_index(drop=True)
    gen = ImageDataGenerator()
    print('{0:70s} for train generator'.format(' '), '\r', end='')
    train_gen = gen.flow_from_dataframe(train_df, x_col='filepaths', y_col=ycol, target_size=img_size, classes=classes,
                                        class_mode='categorical', color_mode='rgb', shuffle=True, batch_size=batch_size)
    print('{0:70s} for valid generator'.format(' '), '\r', end='')
    valid_gen = gen.flow_from_dataframe(valid_df, x_col='filepaths', y_col=ycol, target_size=img_size, classes=classes,
                                        class_mode='categorical', color_mode='rgb', shuffle=False, batch_size=batch_size)
    print('{0:70s} for test generator'.format(' '), '\r', end='')
    test_gen = gen.flow_from_dataframe(test_df, x_col='filepaths', y_col=ycol, target_size=img_size, classes=classes,
                                       class_mode='categorical', color_mode='rgb', shuffle=False, batch_size=batch_size)
    return train_gen, test_gen, valid_gen

In [ ]:
def show_image_samples(gen):
    print_in_color('Below are some example training images')
    classes = list(gen.class_indices.keys())
    images, labels = next(gen)
    plt.figure(figsize=(25, 25))
    r = min(len(labels), 25)
    for i in range(r):
        plt.subplot(5, 5, i + 1)
        plt.imshow(images[i] / 255)
        plt.title(classes[np.argmax(labels[i])], color='blue', fontsize=18)
        plt.axis('off')
    plt.show()

In [ ]:
def F1_score(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    return 2 * (precision * recall) / (precision + recall + K.epsilon())

In [ ]:
def make_model(img_size, class_count, lr, modnum):
    K.clear_session()
    img_shape = (img_size[0], img_size[1], 3)
    builders = {
        1: (EfficientNetV2B0, 'EfficientNetV2B0'),
        2: (EfficientNetV2B1, 'EfficientNetV2B1'),
        3: (EfficientNetV2B2, 'EfficientNetV2B2'),
        4: (EfficientNetV2B3, 'EfficientNetV2B3'),
        5: (EfficientNetV2S, 'EfficientNetV2S'),
        6: (EfficientNetV2M, 'EfficientNetV2M'),
        7: (EfficientNetV2L, 'EfficientNetV2L')}
    builder, model_name = builders[modnum]
    base_model = builder(include_top=False, weights='imagenet', input_shape=img_shape, pooling='max')
    base_model.trainable = True
    x = base_model.output
    x = BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001)(x)
    x = Dense(256, kernel_regularizer=regularizers.l2(l=0.016), activity_regularizer=regularizers.l1(0.006),
              bias_regularizer=regularizers.l1(0.006), activation='relu')(x)
    x = Dropout(rate=0.4, seed=123)(x)
    output = Dense(class_count, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=output)
    model.compile(Adamax(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy', F1_score])
    print(f'Created {model_name} model with initial learning rate set to {lr}')
    return model

In [ ]:
class LR_ASK(keras.callbacks.Callback):
    def __init__(self, model, epochs, ask_epoch, batches, dwell=True, factor=.4):
        super(LR_ASK, self).__init__()
        self.model = model
        self.epochs = epochs
        self.ask_epoch = min(ask_epoch, epochs)
        self.lowest_vloss = np.inf
        self.best_weights = self.model.get_weights()
        self.best_epoch = 1
        self.dwell = dwell
        self.factor = factor
        self.header = True
        self.batches = batches

    def on_train_begin(self, logs=None):
        msg1 = f'Training will proceed until epoch {self.ask_epoch} then you will be asked to\n'
        msg2 = 'enter H to halt training or enter an integer for how many more epochs to run then be asked again'
        print_in_color(msg1 + msg2)
        if self.dwell:
            print_in_color('learning rate will be automatically adjusted during training', (0, 255, 0))
        self.start_time = time.time()

    def on_train_end(self, logs=None):
        print_in_color(f'loading model with weights from epoch {self.best_epoch}', (0, 255, 255))
        self.model.set_weights(self.best_weights)
        tr_duration = time.time() - self.start_time
        hours = int(tr_duration // 3600)
        minutes = int((tr_duration - hours * 3600) // 60)
        seconds = tr_duration - hours * 3600 - minutes * 60
        print_in_color(f'training elapsed time was {hours} hours, {minutes} minutes, {seconds:4.2f} seconds')

    def on_epoch_begin(self, epoch, logs=None):
        self.ep_start = time.time()

    def on_train_batch_end(self, batch, logs=None):
        acc = logs.get('accuracy', 0.0) * 100
        loss = logs.get('loss', 0.0)
        msg = '{0:20s}processing batch {1:} of {2:5s}- accuracy=  {3:5.3f} - loss: {4:8.5f}          '.format(' ', str(batch), str(self.batches), acc, loss)
        print(msg, '\r', end='')

    def on_epoch_end(self, epoch, logs=None):
        if self.header:
            msg = '{0:^7s}{1:^9s}{2:^9s}{3:^9s}{4:^10s}{5:^13s}{6:^10s}{7:^13s}{8:13s}\n'
            msg1 = msg.format('Epoch', 'Train', 'Train', 'Valid', 'Valid', 'V_Loss %', 'Learning', 'Next LR', 'Duration in')
            msg = '{0:^7s}{1:^9s}{2:^9s}{3:^9s}{4:^10s}{5:^13s}{6:^10s}{7:^13s}{8:13s}'
            msg2 = msg.format(' ', 'Loss', 'Accuracy', 'Loss', 'Accuracy', 'Improvement', 'Rate', 'Rate', '  Seconds')
            print_in_color(msg1 + msg2)
            self.header = False
        duration = time.time() - self.ep_start
        vloss = logs.get('val_loss')
        aloss = logs.get('loss')
        acc = logs.get('accuracy')
        v_acc = logs.get('val_accuracy')
        lr = float(K.get_value(self.model.optimizer.learning_rate))
        if epoch > 0:
            pimprov = (self.lowest_vloss - vloss) / self.lowest_vloss * 100
        else:
            pimprov = 0.0
        row = '{0:^7s}{1:^9.4f}{2:^9.2f}{3:^9.4f}{4:^10.2f}{5:^13.2f}{6:^10.6f}{7:11.6f}{8:^15.2f}'
        if vloss < self.lowest_vloss:
            self.lowest_vloss = vloss
            self.best_weights = self.model.get_weights()
            self.best_epoch = epoch + 1
            print_in_color(row.format(str(epoch + 1), aloss, acc * 100, vloss, v_acc * 100, pimprov, lr, lr, duration), (0, 255, 0))
        elif self.dwell:
            new_lr = lr * self.factor
            print_in_color(row.format(str(epoch + 1), aloss, acc * 100, vloss, v_acc * 100, pimprov, lr, new_lr, duration), (255, 255, 0))
            K.set_value(self.model.optimizer.learning_rate, new_lr)
            self.model.set_weights(self.best_weights)
        else:
            print_in_color(row.format(str(epoch + 1), aloss, acc * 100, vloss, v_acc * 100, pimprov, lr, lr, duration), (255, 255, 0))
        if epoch + 1 == self.ask_epoch and epoch + 1 < self.epochs:
            print_in_color('\n Enter H to end training or an integer for the number of additional epochs to run then ask again')
            while True:
                ans = input().strip()
                if ans.lower() == 'h' or ans == '0':
                    print_in_color(f'you entered {ans}, training halted on epoch {epoch + 1} due to user input\n')
                    self.model.stop_training = True
                    break
                try:
                    extra = int(ans)
                except ValueError:
                    print_in_color('invalid input, enter H or a positive integer')
                    continue
                if extra < 1:
                    print_in_color('invalid input, enter H or a positive integer')
                    continue
                self.header = True
                self.ask_epoch = min(self.ask_epoch + extra, self.epochs)
                print_in_color(f'you entered {ans} training will continue to epoch {self.ask_epoch}')
                if not self.dwell:
                    current_lr = float(K.get_value(self.model.optimizer.learning_rate))
                    print_in_color(f'current LR is {current_lr:8.6f} hit enter to keep this LR or enter a new LR')
                    while True:
                        lr_ans = input(' ').strip()
                        if lr_ans == '':
                            print_in_color(f'keeping current LR of {current_lr:7.5f}')
                            break
                        try:
                            new_lr = float(lr_ans)
                        except ValueError:
                            print_in_color('invalid learning rate')
                            continue
                        K.set_value(self.model.optimizer.learning_rate, new_lr)
                        print_in_color(f'changing LR to {lr_ans}')
                        break
                break

In [ ]:
def tr_plot(tr_data):
    start_epoch = 0
    tacc = tr_data.history['accuracy']
    tloss = tr_data.history['loss']
    vacc = tr_data.history['val_accuracy']
    vloss = tr_data.history['val_loss']
    tr_f1 = tr_data.history['F1_score']
    vf1 = tr_data.history['val_F1_score']
    Epoch_count = len(tacc) + start_epoch
    Epochs = [i + 1 for i in range(start_epoch, Epoch_count)]
    index_loss = int(np.argmin(vloss))
    val_lowest = vloss[index_loss]
    index_acc = int(np.argmax(vacc))
    acc_highest = vacc[index_acc]
    index_vf1 = int(np.argmax(vf1))
    vf1_highest = vf1[index_vf1]
    plt.style.use('fivethirtyeight')
    sc_label = 'best epoch= ' + str(index_loss + 1 + start_epoch)
    vc_label = 'best epoch= ' + str(index_acc + 1 + start_epoch)
    f1_label = 'best epoch= ' + str(index_vf1 + 1 + start_epoch)
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(25, 10))
    axes[0].plot(Epochs, tloss, 'r', label='Training loss')
    axes[0].plot(Epochs, vloss, 'g', label='Validation loss')
    axes[0].scatter(index_loss + 1 + start_epoch, val_lowest, s=150, c='blue', label=sc_label)
    axes[0].scatter(Epochs, tloss, s=100, c='red')
    axes[0].set_title('Training and Validation Loss')
    axes[0].set_xlabel('Epochs', fontsize=18)
    axes[0].set_ylabel('Loss', fontsize=18)
    axes[0].legend()
    axes[1].plot(Epochs, tacc, 'r', label='Training Accuracy')
    axes[1].scatter(Epochs, tacc, s=100, c='red')
    axes[1].plot(Epochs, vacc, 'g', label='Validation Accuracy')
    axes[1].scatter(index_acc + 1 + start_epoch, acc_highest, s=150, c='blue', label=vc_label)
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].set_xlabel('Epochs', fontsize=18)
    axes[1].set_ylabel('Accuracy', fontsize=18)
    axes[1].legend()
    axes[2].plot(Epochs, tr_f1, 'r', label='Training F1 score')
    axes[2].plot(Epochs, vf1, 'g', label='Validation F1 score')
    axes[2].scatter(index_vf1 + 1 + start_epoch, vf1_highest, s=150, c='blue', label=f1_label)
    axes[2].scatter(Epochs, tr_f1, s=100, c='red')
    axes[2].set_title('Training and Validation F1 score')
    axes[2].set_xlabel('Epochs', fontsize=18)
    axes[2].set_ylabel('F1  score', fontsize=18)
    axes[2].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def save_history_to_csv(history, csvpath):
    trdict = history.history
    df = pd.DataFrame()
    df['Epoch'] = list(np.arange(1, len(trdict['loss']) + 1))
    for key in trdict.keys():
        df[key] = list(trdict[key])
    df.to_csv(csvpath, index=False)

In [ ]:
def predictor(model, test_gen):
    classes = list(test_gen.class_indices.keys())
    class_count = len(classes)
    preds = model.predict(test_gen, verbose=1)
    ytrue = np.array(test_gen.labels)
    ypred = np.argmax(preds, axis=1)
    test_count = len(ypred)
    wrong = np.where(ytrue != ypred)[0]
    errors = len(wrong)
    accuracy = (test_count - errors) * 100 / test_count
    f1score = f1_score(ytrue, ypred, average='weighted') * 100
    print(f'There were {errors} errors in {test_count} tests for an accuracy of {accuracy:6.2f} and an F1 score of {f1score:6.2f}')
    plt.style.use('fivethirtyeight')
    if errors > 0:
        mis_counter = Counter(classes[ytrue[i]] for i in wrong)
        mis_classes = sorted(mis_counter.keys())
        mis_values = [mis_counter[k] for k in mis_classes]
        plt.figure(figsize=(10, max(3, len(mis_classes) / 2)))
        plt.barh(mis_classes, mis_values)
        plt.title('Classification Errors on Test Set by Class', fontsize=20, color='blue')
        plt.xlabel('NUMBER OF MISCLASSIFICATIONS', fontsize=20, color='blue')
        plt.ylabel('CLASS', fontsize=20, color='blue')
        plt.show()
    if class_count <= 30:
        labels = np.arange(class_count)
        cm = confusion_matrix(ytrue, ypred, labels=labels)
        plt.figure(figsize=(12, 8))
        sns.heatmap(cm, annot=True, vmin=0, fmt='g', cmap='Blues', cbar=False)
        plt.xticks(np.arange(class_count) + .5, classes, rotation=90)
        plt.yticks(np.arange(class_count) + .5, classes, rotation=0)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Confusion Matrix')
        plt.show()
        clr = classification_report(ytrue, ypred, labels=labels, target_names=classes, digits=4, zero_division=0)
        print('Classification Report:\n----------------------\n', clr)
    return f1score

In [ ]:
def save_model(model, subject, classes, img_size, f1score, working_dir):
    name = f'{subject}-{len(classes)}-({img_size[0]} X {img_size[1]})- {f1score:5.2f}.h5'
    model_save_loc = os.path.join(working_dir, name)
    try:
        model.save(model_save_loc)
        print_in_color(f'model was saved as {model_save_loc}', (0, 255, 255), (0, 0, 0))
    except Exception as e:
        print_in_color(f'model could not be saved: {e}', (0, 255, 255), (0, 0, 0))

In [ ]:
def run(working_dir=None, train_dir=None, test_dir=None, valid_dir=None, start=True):
    if start:
        print_in_color('enter the full path to the working directory where data will be stored.')
        working_dir = input(' ').strip()
        print_in_color('Enter the full path to the train directory')
        train_dir = input(' ').strip()
        print_in_color('Enter the full path to the validation directory. If there is no validation directory press enter')
        ans = input(' ').strip()
        valid_dir = None if ans == '' else ans
        print_in_color('Enter the full path to the test directory. If there is no test directory press enter')
        ans = input(' ').strip()
        test_dir = None if ans == '' else ans
    os.makedirs(working_dir, exist_ok=True)
    ftotal, flargest, maxclass, fsmallest, minclass = check_dataset_size(train_dir)
    msg1 = f'the train directory contains {ftotal} files, class {maxclass} has the most images of {flargest} files\n'
    msg2 = f'class {minclass} contains the least number of images of {fsmallest} files\n'
    msg3 = 'NOTE if the value of most images is the same as the value of least images the dataset is balanced\n'
    msg4 = 'When dealing with very large data sets to save time you may not want to read in all the image files\n'
    msg5 = 'to limit the maximum number of image files in any class you can enter an integer limiter value'
    print_in_color(msg1 + msg2 + msg3 + msg4 + msg5)
    print_in_color('input a limiter integer value to limit max number of images in a class, for no limit hit enter')
    limiter = ask_int(default=None, allow_empty=True, minimum=1)
    if limiter is None:
        print_in_color('dataset will be processed as is with no limiter')
    else:
        print_in_color(f'images will be limited to a maximum of {limiter} images in each class')
    print_in_color('enter the minimum number of training images a class must have to be included in the dataset or press enter to use the default value of 10')
    min_images = ask_int(default=10, allow_empty=True, minimum=1)
    train_df, test_df, valid_df, classes, class_count, max_samples, min_samples, have, wave = make_dataframes(train_dir, test_dir, valid_dir, limiter, min_images)
    print_in_color(f'the average height of training images is {have} and average width is {wave}\n  enter the image height to be used to train the model')
    img_height = ask_int(minimum=32)
    print_in_color(' Enter the image width to be used to train the model')
    img_width = ask_int(minimum=32)
    img_size = (img_height, img_width)
    print_in_color(f'model will be trained with image shape of  ( {img_height}, {img_width} )')
    print_in_color('enter A to auto balance the train set or hit enter to leave train set unchanged')
    ans = input(' ').strip()
    if ans.lower() == 'a':
        print_in_color('enter the number of images you want to have in each class of the train data set')
        max_images = ask_int(minimum=1)
        train_df, classes, class_count = trim(train_df, max_images, min_images, 'labels')
        train_df = balance(train_df, max_images, 'labels', working_dir)
        plot_label_count(train_df, 'Images per Label after Auto Balance of train data set')
    else:
        print_in_color('training data set will be used as is')
    classes = sorted(train_df['labels'].unique())
    class_count = len(classes)
    print_in_color('enter the desired batch size or press enter to use the default batch size of 20')
    bs = ask_int(default=20, allow_empty=True, minimum=1)
    print('select a model number from the list below ', flush=True)
    print('{0:^9s}{1:^20s}{2:^12s}'.format('Model No', 'Model Type', 'Parameters'), flush=True)
    models = ['EfficientNetV2B0', 'EfficientNetV2B1', 'EfficientNetV2B2', 'EfficientNetV2B3', 'EfficientNetV2S', 'EfficientNetV2M', 'EfficientNetV2L']
    parameters = ['7.2 M', '8.2 M', '10.2 M', '14.5 M', '21.6 M', '54.4 M', '119.0 M']
    for i in range(7):
        print('{0:^9s}{1:^20s}{2:^12s}'.format(str(i + 1), models[i], parameters[i]))
    modnum = ask_int(minimum=1, maximum=7)
    lr = .001
    train_gen, test_gen, valid_gen = make_gens(bs, 'labels', train_df, test_df, valid_df, img_size, classes)
    show_image_samples(train_gen)
    model = make_model(img_size, class_count, lr, modnum)
    print('enter the number of epochs to run initially. After these epochs complete you can enter H to halt training')
    print('or enter an integer for how many more epochs to run then be asked again')
    ask_epoch = ask_int(minimum=1)
    epochs = 1000
    batches = int(np.ceil(len(train_df) / bs))
    ask = LR_ASK(model, epochs=epochs, ask_epoch=ask_epoch, batches=batches)
    history = model.fit(x=train_gen, epochs=epochs, verbose=0, callbacks=[ask], validation_data=valid_gen, initial_epoch=0)
    tr_plot(history)
    print_in_color('To save the training data to a csv file enter the name for the csv file or press enter to not save the data')
    ans = input(' ').strip()
    if ans != '':
        csvpath = os.path.join(working_dir, ans + '.csv')
        save_history_to_csv(history, csvpath)
        print_in_color(f'training data saved to {csvpath}')
    f1score = predictor(model, test_gen)
    print_in_color(f'your trained model will be saved to directory {working_dir} enter a subject for the saved model')
    subject = input(' ').strip()
    if subject == '':
        subject = 'model'
    save_model(model, subject, classes, img_size, f1score, working_dir)
    print_in_color('model save nomenclature is directory/subject-number of classes- (img_height, img_width)- F1score.h5')
    if os.path.isdir(os.path.join(working_dir, 'aug')):
        shutil.rmtree(os.path.join(working_dir, 'aug'))
    return working_dir, train_dir, test_dir, valid_dir, model

In [ ]:
working_dir, train_dir, test_dir, valid_dir, model = run(start=True)
while True:
    print_in_color('enter R to rerun the classifier or press enter to quit')
    ans = input(' ').strip()
    if ans.lower() != 'r':
        print_in_color('process completed')
        break
    print_in_color('press Enter to use original directory values or enter any key to input new ones')
    ans = input(' ').strip()
    if ans == '':
        working_dir, train_dir, test_dir, valid_dir, model = run(working_dir, train_dir, test_dir, valid_dir, start=False)
    else:
        working_dir, train_dir, test_dir, valid_dir, model = run(start=True)